# 타이타닉 생존 예측 실습
### 목표 : 전처리 방법 변경 및 모델을 Tensorflow 딥러닝 모델로 변경하여 제출 후 스코어 0.8 이상 도달하기

### baseline

In [ ]:
# 필요한 라이브러리 임포트
import pandas as pd  # 표 형태 데이터를 다루는 라이브러리입니다.
import numpy as np  # 배열과 수치 연산 라이브러리입니다.
from sklearn.model_selection import train_test_split  # 훈련/평가 데이터 분리 함수입니다.
from sklearn.ensemble import RandomForestClassifier  # 랜덤포레스트 분류 모델입니다.
from sklearn.metrics import accuracy_score, classification_report  # 정확도 지표 함수, 분류 지표 요약 함수입니다.

# 데이터 불러오기
train = pd.read_csv('train.csv')  # CSV 파일을 데이터프레임으로 읽습니다.
test = pd.read_csv('test.csv')  # CSV 파일을 데이터프레임으로 읽습니다.

# 데이터 전처리
def preprocess_data(df):
    # 결측치 처리
    df['Age'] = df['Age'].fillna(df['Age'].mean())  # 결측치를 지정한 값으로 채웁니다.
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])  # 결측치를 지정한 값으로 채웁니다.
    df['Fare'] = df['Fare'].fillna(df['Fare'].mean())  # 결측치를 지정한 값으로 채웁니다.
    
    # 범주형 변수 처리
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})  # 값을 지정한 규칙으로 바꿉니다.
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})  # 값을 지정한 규칙으로 바꿉니다.
    
    # 필요한 특성 선택
    features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
    return df[features]

# 학습 데이터 전처리
X = preprocess_data(train)
y = train['Survived']

# 학습 데이터와 검증 데이터 분리
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 훈련/평가용으로 나눕니다.

# RandomForest 모델 생성 및 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)  # 랜덤포레스트입니다. n_estimators는 트리 수입니다.
rf_model.fit(X_train, y_train)  # 데이터로 모델이나 변환 기준을 학습합니다.

# 검증 데이터로 예측
val_pred = rf_model.predict(X_val)  # 학습한 모델로 새 값을 예측합니다.

# 모델 성능 평가
print('검증 데이터 정확도:', accuracy_score(y_val, val_pred))  # 정확도 비율을 계산합니다.
print('\n분류 보고서:')  # 문자열을 정수로 바꿉니다.
print(classification_report(y_val, val_pred))  # 정밀도/재현율/F1을 요약합니다.

# 테스트 데이터 예측
test_processed = preprocess_data(test)
test_pred = rf_model.predict(test_processed)  # 학습한 모델로 새 값을 예측합니다.

# 제출 파일 생성
submission = pd.DataFrame({  # 데이터프레임을 직접 만듭니다.
    'PassengerId': test['PassengerId'],
    'Survived': test_pred
})
submission.to_csv('submission.csv', index=False)
print('\n제출 파일이 생성되었습니다.')  # 문자열을 정수로 바꿉니다.
